Importación de paquetes

In [1]:
import random
import torch
import csv
import cv2

from PIL import Image
from torch.utils.data import DataLoader
from transformers import AutoProcessor, AutoModelForImageTextToText
from collections import defaultdict
from pathlib import Path
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

c:\Users\Dylan\anaconda3\envs\P4_vlm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1113 11:03:23.180000 3264 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Entrenamiento del modelo SmolVLM

In [ ]:
DATA_DIR = "plates"
MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
EPOCHS = 20
BATCH_SIZE = 8
LR = 5e-5
IMG_SIZE = 512
OUT_DIR = "results/plate-ocr-smol"
PROMPT = "Lee la matrícula de la imagen."

device = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
autocast = lambda: torch.autocast(device, dtype=torch.bfloat16, enabled=use_bf16)

def cargar_lista(split):
    with open(f"{DATA_DIR}/{split}.txt", encoding="utf-8") as f:
        return [l.rstrip("\n").split("\t") for l in f if l.strip()]

N_VAL = len(cargar_lista("val"))

train_data, val_data = cargar_lista("train"), cargar_lista("val")[:N_VAL]
print(f"train: {len(train_data)} | val: {len(val_data)} | device: {device} | bf16: {use_bf16}")

# Modelo y procesador
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.image_processor.do_image_splitting = False
processor.image_processor.size = {"longest_edge": IMG_SIZE}
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(device)

# Congelar el encoder visual
for p in model.model.vision_model.parameters():
    p.requires_grad = False

mensajes = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
prompt_txt = processor.apply_chat_template(mensajes, add_generation_prompt=True)
EOS = "<end_of_utterance>"

def abrir(ruta):
    return Image.open(f"{DATA_DIR}/{ruta}").convert("RGB")

# Longitud del prompt (constante al no haber image splitting) para enmascarar la loss
prompt_len = processor(text=prompt_txt, images=[abrir(train_data[0][0])],
                       return_tensors="pt")["input_ids"].shape[1]

def collate_train(batch):
    processor.tokenizer.padding_side = "right"
    imgs = [[abrir(r)] for r, _ in batch]
    textos = [f"{prompt_txt}{t}{EOS}" for _, t in batch]
    enc = processor(text=textos, images=imgs, padding=True, return_tensors="pt")
    labels = enc["input_ids"].clone()
    labels[:, :prompt_len] = -100
    labels[enc["attention_mask"] == 0] = -100
    enc["labels"] = labels
    return enc

@torch.no_grad()
def evaluar():
    model.eval()
    processor.tokenizer.padding_side = "left"
    aciertos = 0
    for i in range(0, len(val_data), BATCH_SIZE):
        lote = val_data[i:i + BATCH_SIZE]
        enc = processor(text=[prompt_txt] * len(lote), images=[[abrir(r)] for r, _ in lote],
                        padding=True, return_tensors="pt").to(device)
        with autocast():
            out = model.generate(**enc, max_new_tokens=12, do_sample=False)
        preds = processor.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        aciertos += sum(p.strip().replace(" ", "").upper() == t for p, (_, t) in zip(preds, lote))
    model.train()
    return aciertos / len(val_data)

# Entrenamiento
loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                    collate_fn=collate_train, num_workers=0)
optim = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
total = EPOCHS * len(loader)
sched = torch.optim.lr_scheduler.LambdaLR(
    optim, lambda s: min(1, (s + 1) / (0.05 * total)) * max(0.0, 1 - s / total))

best_acc, step = -1.0, 0
model.train()
for epoch in range(1, EPOCHS + 1):
    perdida = 0.0
    for i, batch in enumerate(loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        with autocast():
            loss = model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step(); sched.step(); optim.zero_grad(set_to_none=True)
        perdida += loss.item(); step += 1
        if i % 50 == 0:
            print(f"  epoch {epoch} [{i}/{len(loader)}] loss {perdida / i:.4f}")

    acc = evaluar()
    print(f"Epoch {epoch}/{EPOCHS} | loss {perdida / len(loader):.4f} | val acc (exacta) {acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        model.save_pretrained(f"{OUT_DIR}/best_model")
        processor.save_pretrained(f"{OUT_DIR}/best_model")
        print(f"  -> nuevo mejor modelo guardado ({acc:.4f})")

print("Listo. Modelo en", f"{OUT_DIR}/best_model")

train: 2110 | val: 247 | device: cuda | bf16: True


Loading weights: 100%|██████████| 471/471 [00:00<00:00, 858.24it/s]


  epoch 1 [50/264] loss 1.6286
  epoch 1 [100/264] loss 1.0175
  epoch 1 [150/264] loss 0.7596
  epoch 1 [200/264] loss 0.6576
  epoch 1 [250/264] loss 0.6037
Epoch 1/20 | loss 0.5819 | val acc (exacta) 0.7854


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.17s/it]


  -> nuevo mejor modelo guardado (0.7854)
  epoch 2 [50/264] loss 0.2507
  epoch 2 [100/264] loss 0.2373
  epoch 2 [150/264] loss 0.2500
  epoch 2 [200/264] loss 0.2424
  epoch 2 [250/264] loss 0.2427
Epoch 2/20 | loss 0.2513 | val acc (exacta) 0.8340


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.51s/it]


  -> nuevo mejor modelo guardado (0.8340)
  epoch 3 [50/264] loss 0.1433
  epoch 3 [100/264] loss 0.1632
  epoch 3 [150/264] loss 0.1660
  epoch 3 [200/264] loss 0.1705
  epoch 3 [250/264] loss 0.1757
Epoch 3/20 | loss 0.1718 | val acc (exacta) 0.8502


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.33s/it]


  -> nuevo mejor modelo guardado (0.8502)
  epoch 4 [50/264] loss 0.0934
  epoch 4 [100/264] loss 0.1029
  epoch 4 [150/264] loss 0.1102
  epoch 4 [200/264] loss 0.1110
  epoch 4 [250/264] loss 0.1122
Epoch 4/20 | loss 0.1134 | val acc (exacta) 0.8381
  epoch 5 [50/264] loss 0.0767
  epoch 5 [100/264] loss 0.0764
  epoch 5 [150/264] loss 0.0708
  epoch 5 [200/264] loss 0.0770
  epoch 5 [250/264] loss 0.0774
Epoch 5/20 | loss 0.0772 | val acc (exacta) 0.8340
  epoch 6 [50/264] loss 0.0445
  epoch 6 [100/264] loss 0.0455
  epoch 6 [150/264] loss 0.0511
  epoch 6 [200/264] loss 0.0507
  epoch 6 [250/264] loss 0.0522
Epoch 6/20 | loss 0.0531 | val acc (exacta) 0.8462
  epoch 7 [50/264] loss 0.0417
  epoch 7 [100/264] loss 0.0329
  epoch 7 [150/264] loss 0.0352
  epoch 7 [200/264] loss 0.0370
  epoch 7 [250/264] loss 0.0387
Epoch 7/20 | loss 0.0373 | val acc (exacta) 0.8300
  epoch 8 [50/264] loss 0.0238
  epoch 8 [100/264] loss 0.0279
  epoch 8 [150/264] loss 0.0264
  epoch 8 [200/264] los

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.33s/it]


  -> nuevo mejor modelo guardado (0.8583)
  epoch 9 [50/264] loss 0.0110
  epoch 9 [100/264] loss 0.0113
  epoch 9 [150/264] loss 0.0117
  epoch 9 [200/264] loss 0.0127
  epoch 9 [250/264] loss 0.0130
Epoch 9/20 | loss 0.0136 | val acc (exacta) 0.8421
  epoch 10 [50/264] loss 0.0076
  epoch 10 [100/264] loss 0.0064
  epoch 10 [150/264] loss 0.0061
  epoch 10 [200/264] loss 0.0063
  epoch 10 [250/264] loss 0.0071
Epoch 10/20 | loss 0.0072 | val acc (exacta) 0.8421
  epoch 11 [50/264] loss 0.0040
  epoch 11 [100/264] loss 0.0038
  epoch 11 [150/264] loss 0.0032
  epoch 11 [200/264] loss 0.0035
  epoch 11 [250/264] loss 0.0031
Epoch 11/20 | loss 0.0033 | val acc (exacta) 0.8421
  epoch 12 [50/264] loss 0.0009
  epoch 12 [100/264] loss 0.0007
  epoch 12 [150/264] loss 0.0014
  epoch 12 [200/264] loss 0.0014
  epoch 12 [250/264] loss 0.0016
Epoch 12/20 | loss 0.0016 | val acc (exacta) 0.8462
  epoch 13 [50/264] loss 0.0003
  epoch 13 [100/264] loss 0.0004
  epoch 13 [150/264] loss 0.0005
  

Prueba del modelo SmolVLM en vídeo

In [2]:
OUT_DIR = Path("results") / "plate-ocr"
VIDEO_ORIG = "video_test.MP4"
CSV_IN = OUT_DIR / "tracking.csv"
VIDEO_OUT = OUT_DIR.parent / "plate-ocr-smol" / "video_final.mp4"
CSV_OUT = OUT_DIR.parent / "plate-ocr-smol" / "tracking_final.csv"
MODEL_DIR = "results/plate-ocr-smol/best_model"
PROMPT = "Lee la matrícula de la imagen."
PAD = 0.05

device = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
autocast = lambda: torch.autocast(device, dtype=torch.bfloat16, enabled=use_bf16)

# Cargar el lector entrenado
processor = AutoProcessor.from_pretrained(MODEL_DIR)
model = AutoModelForImageTextToText.from_pretrained(MODEL_DIR, torch_dtype=torch.float32).to(device)
model.eval()

mensajes = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
prompt_txt = processor.apply_chat_template(mensajes, add_generation_prompt=True)

# Convierte el recorte BGR de OpenCV a PIL y decodifica con SmolVLM
@torch.no_grad()
def leer_matricula(crop):
    img = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    enc = processor(text=prompt_txt, images=[img], return_tensors="pt").to(device)
    with autocast():
        out = model.generate(**enc, max_new_tokens=12, do_sample=False)
    texto = processor.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    return texto.strip().replace(" ", "").upper()

# Texto verde sobre fondo negro para que se lea bien
def poner_texto(img, texto, x, y):
    (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
    y = max(y, th + 6)
    cv2.rectangle(img, (x, y - th - 6), (x + tw + 4, y + 2), (0, 0, 0), -1)
    cv2.putText(img, texto, (x + 2, y - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

# Leer el CSV de YOLO y agrupar filas por fotograma
with open(CSV_IN, newline="", encoding="utf-8") as f:
    filas = list(csv.DictReader(f))
por_frame = defaultdict(list)
for fila in filas:
    por_frame[int(fila["fotograma"])].append(fila)

cap = cv2.VideoCapture(VIDEO_ORIG)
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir {VIDEO_ORIG}")
fps = cap.get(cv2.CAP_PROP_FPS) or 25
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
VIDEO_OUT.parent.mkdir(parents=True, exist_ok=True)
writer = cv2.VideoWriter(str(VIDEO_OUT), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

n = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    for fila in por_frame.get(n, []):
        x1, y1, x2, y2 = (int(fila[k]) for k in ("x1", "y1", "x2", "y2"))
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 128, 0), 2)
        cv2.putText(frame, f"{fila['tipo_objeto']} #{fila['id_tracking']}", (x1, max(y1 - 8, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 128, 0), 2)

        if not fila["mx1"]:
            continue
        mx1, my1, mx2, my2 = (int(fila[k]) for k in ("mx1", "my1", "mx2", "my2"))
        bw, bh = mx2 - mx1, my2 - my1
        crop = frame[max(my1 - int(bh * PAD), 0):min(my2 + int(bh * PAD), H),
                     max(mx1 - int(bw * PAD), 0):min(mx2 + int(bw * PAD), W)].copy()
        cv2.rectangle(frame, (mx1, my1), (mx2, my2), (0, 255, 0), 2)
        if crop.shape[1] >= 20 and crop.shape[0] >= 8:
            fila["texto_matricula"] = leer_matricula(crop)
            poner_texto(frame, fila["texto_matricula"], mx1, my1 - 6)

    writer.write(frame)
    n += 1

cap.release()
writer.release()

with open(CSV_OUT, "w", newline="", encoding="utf-8") as f:
    wr = csv.DictWriter(f, fieldnames=list(filas[0].keys()))
    wr.writeheader()
    wr.writerows(filas)

print("Vídeo:", VIDEO_OUT)
print("CSV final:", CSV_OUT)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 471/471 [00:00<00:00, 711.56it/s] 


Vídeo: results\plate-ocr-smol\video_final.mp4
CSV final: results\plate-ocr-smol\tracking_final.csv
